# 05.06 · Running models locally

**Chapter:** Working with LLM APIs · **Milestone:** M2

## Goals
- [ ] Run models with Ollama (CLI, native API, OpenAI-compatible API) and control context length / options
- [ ] Run the same model family with MLX-LM on Apple Silicon and compare speed
- [ ] Know LM Studio and llama.cpp, and when each fits
- [ ] Estimate which models fit your machine (quantization basics)

## Resources
- [Ollama](https://ollama.com/) · [Ollama API docs](https://docs.ollama.com/api) — free
- [MLX-LM](https://github.com/ml-explore/mlx-lm) · [mlx-community models](https://huggingface.co/mlx-community) — free
- [llama.cpp](https://github.com/ggml-org/llama.cpp) · [LM Studio](https://lmstudio.ai/) — free

| Tool | What it is | Best for |
|---|---|---|
| **Ollama** | daemon + CLI + model library, wraps llama.cpp | easiest; OpenAI-compatible API on `:11434` |
| **LM Studio** | desktop GUI; llama.cpp + MLX engines | exploring models visually; local server on `:1234` |
| **llama.cpp** | the C++ engine itself (`llama-server`, GGUF format) | maximum control, any hardware, grammars |
| **MLX-LM** | Apple's framework for Apple Silicon | fastest on Macs; fine-tuning on Mac (chapter 12) |
| **vLLM / SGLang** | GPU servers with batching | serving many users on NVIDIA GPUs (chapter 13) |

In [ ]:
import platform
import subprocess
import time

import httpx

from aieng.llm import client

OLLAMA = "http://localhost:11434"
MODEL = "qwen3:1.7b"

## 1. Ollama CLI
In a terminal: `ollama pull qwen3:1.7b`, `ollama run qwen3:1.7b`, `ollama list`, `ollama ps` (what's loaded in memory), `ollama rm`.

In [ ]:
print(subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout)

## 2. Native API: options and real speed numbers
The native `/api/chat` endpoint exposes options the OpenAI-compatible one doesn't, e.g. **`num_ctx`** (context length — Ollama's
default is small; long prompts get silently truncated!), and returns precise timing: `eval_count` tokens in `eval_duration` ns.

In [ ]:
r = httpx.post(f"{OLLAMA}/api/chat", timeout=300, json={
    "model": MODEL,
    "messages": [{"role": "user", "content": "Explain what quantization is in 80 words."}],
    "stream": False,
    "think": False,  # qwen3 reasoning off
    "options": {"num_ctx": 8192, "temperature": 0.7},
}).json()
print(r["message"]["content"])
print(f"\nprompt: {r['prompt_eval_count']} tok in {r['prompt_eval_duration'] / 1e9:.2f}s | "
      f"output: {r['eval_count']} tok at {r['eval_count'] / (r['eval_duration'] / 1e9):.1f} tok/s | load: {r['load_duration'] / 1e9:.2f}s")

## 3. OpenAI-compatible API
The same server speaks the OpenAI format at `/v1` — that's what `aieng.llm` uses, so every lesson works locally.

In [ ]:
oc, _ = client("ollama")
print(oc.chat.completions.create(model=MODEL, reasoning_effort="none",
                                 messages=[{"role": "user", "content": "Say 'hello' in 5 languages."}]).choices[0].message.content)

## 4. MLX-LM on Apple Silicon
Downloads a 4-bit model from `mlx-community` on first run (~0.4 GB for Qwen3-0.6B, ~1 GB for 1.7B).

In [ ]:
if platform.machine() == "arm64" and platform.system() == "Darwin":
    from mlx_lm import generate, load

    mlx_model, mlx_tok = load("mlx-community/Qwen3-1.7B-4bit")
    messages = [{"role": "user", "content": "Explain what quantization is in 80 words."}]
    prompt = mlx_tok.apply_chat_template(messages, add_generation_prompt=True, enable_thinking=False)
    start = time.perf_counter()
    out = generate(mlx_model, mlx_tok, prompt=prompt, max_tokens=200, verbose=False)
    secs = time.perf_counter() - start
    n = len(mlx_tok.encode(out))
    print(out, f"\n\nMLX: {n} tokens in {secs:.1f}s ≈ {n / secs:.0f} tok/s (includes prompt processing)")
else:
    print("MLX needs Apple Silicon — skip")

MLX-LM also has a server: `uv run mlx_lm.server --model mlx-community/Qwen3-1.7B-4bit` → OpenAI-compatible on `:8080`.

## 5. LM Studio and llama.cpp (reference)
```bash
# LM Studio: GUI → download a model → "Developer" tab → Start server  → http://localhost:1234/v1
# llama.cpp:
brew install llama.cpp
llama-server -hf Qwen/Qwen3-1.7B-GGUF:Q4_K_M --port 8081   # downloads GGUF from the Hub, serves OpenAI-compatible API
```
Then in Python: `OpenAI(base_url="http://localhost:8081/v1", api_key="x")`. Same code as every other provider.

## 6. What fits on my machine?
Memory ≈ parameters × bits/8 (+ ~10–30% for KV cache and runtime). On a Mac, the GPU shares system RAM (unified memory),
and macOS lets it use roughly 65–75% by default.

In [ ]:
import os

ram_gb = int(subprocess.run(["sysctl", "-n", "hw.memsize"], capture_output=True, text=True).stdout or 0) / 1e9 if platform.system() == "Darwin" else (os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES")) / 1e9
budget = ram_gb * 0.65
print(f"RAM {ram_gb:.0f} GB -> ~{budget:.0f} GB usable for a model")
for params in [1.7, 4, 8, 14, 32, 70]:
    for bits, name in [(4, "Q4"), (8, "Q8"), (16, "bf16")]:
        need = params * bits / 8 * 1.2
        print(f"{params:>4}B {name:4}: {need:5.1f} GB {'✓' if need < budget else '✗'}", end="   ")
    print()

Q4 (4-bit, e.g. GGUF `Q4_K_M`) is the usual sweet spot: ~4× smaller than bf16 with a small quality loss. Chapter 13 measures that loss.

## Exercises — Milestone M2 check
1. Pull a bigger model that fits (e.g. `qwen3:8b` or `gemma3:12b` depending on RAM). Compare tokens/s and answer quality against `qwen3:1.7b`
   on 5 prompts of your choice.
2. Same model, Ollama vs MLX: measure output tokens/s for a 500-token answer, 3 runs each. Which is faster on your Mac?
3. Set `num_ctx` to 2048 and send a 4,000-token prompt with a question about its *first* paragraph. What happens? Then fix it.
4. **M2 deliverable**: your terminal chat from 05.02 + structured outputs (05.03) + retries/cost log (05.05), working against
   both a hosted provider and local Ollama by changing one env var.

In [ ]:
# your code here